In [2]:
"""
╔══════════════════════════════════════════════════════════════════╗
║                    F L U X  — World Model + RL                  ║
╚══════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  NATURAL LANGUAGE RULES  (observation space X — source text)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FLUX is a two‑player game played on a row of five numbers that
  starts as [2, 1, 3, 1, 2].

  Players alternate turns. On your turn you must choose one cell
  and either Amplify it (×2) or Drain it (÷2, round down). After
  the operation, any cell that reaches 0 is removed and the row
  contracts.

  The player who causes the row to have **exactly one cell** wins
  immediately. However, if the move makes the **total sum exceed 20**,
  that player loses immediately.

  If neither player has won after 15 total moves, a tiebreak applies:
  if the row has fewer than 3 cells the Shrinker (Player 0) wins,
  otherwise the Amplifier (Player 1) wins.

  The Shrinker always goes first.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  EXTRACTED WORLD MODEL  (latent state space S)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  State  :  s  = (cells: list[int], move_number: int)
  Action :  a  = (cell_index: int, op: 'amplify' | 'drain')

  Transition  f(s, a) -> (s', outcome, reward_shrinker, reward_amplifier):

    Rule 1 – Apply operation:
      amplify : cells[i] <- cells[i] * 2
      drain   : cells[i] <- cells[i] // 2

    Rule 2 – Contract row:
      cells <- [c for c in cells if c > 0]

    Rule 3 – Evaluate terminal conditions (priority order):
      (a) sum(cells) > 20   -> mover loses (opponent wins)
      (b) len(cells) == 1   -> mover wins
      (c) move_number >= 15 -> tiebreak: Shrinker wins if len(cells) < 3,
                               otherwise Amplifier wins
      (d) otherwise         -> 'ongoing'

    Rule 4 – Advance turn:
      turn <- 1 - turn

  Guards:
    cell_index in [0, len(cells)-1]
    op in {'amplify', 'drain'}
"""

import json
import os
import random
import time
from collections import defaultdict
from copy import deepcopy

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ════════════════════════════════════════════════════════════════
#  WORLD MODEL
# ════════════════════════════════════════════════════════════════

INITIAL_CELLS  = [2, 1, 3, 1, 2]
SUM_CAP        = 20
MAX_MOVES      = 15
TIEBREAK_CELLS = 3
SHRINKER       = 0
AMPLIFIER      = 1


def initial_state():
    return {"cells": list(INITIAL_CELLS), "move_num": 0}


def current_player(state):
    return state["move_num"] % 2


def valid_actions(state):
    n = len(state["cells"])
    return [(i, op) for i in range(n) for op in ("amplify", "drain")]


def transition(state, action):
    """
    Core world model: f(s, a) -> (s', outcome, reward_shrinker, reward_amplifier)
    Correctly implements the original game rules.
    """
    cells    = list(state["cells"])
    move_num = state["move_num"]
    idx, op  = action
    mover    = current_player(state)   # who is making this move

    # Rule 1
    if op == "amplify":
        cells[idx] *= 2
    else:
        cells[idx] = cells[idx] // 2

    # Rule 2
    cells = [c for c in cells if c > 0]
    move_num += 1
    total = sum(cells) if cells else 0

    # Rule 3 – evaluate terminal conditions
    if total > SUM_CAP:
        # mover loses → opponent wins
        if mover == SHRINKER:
            outcome = "amplifier_wins"
            rs, ra = -1.0, +1.0
        else:
            outcome = "shrinker_wins"
            rs, ra = +1.0, -1.0
    elif len(cells) == 1:
        # mover wins
        if mover == SHRINKER:
            outcome = "shrinker_wins"
            rs, ra = +1.0, -1.0
        else:
            outcome = "amplifier_wins"
            rs, ra = -1.0, +1.0
    elif move_num >= MAX_MOVES:
        # tiebreak
        if len(cells) < TIEBREAK_CELLS:
            outcome = "shrinker_wins"
            rs, ra = +1.0, -1.0
        else:
            outcome = "amplifier_wins"
            rs, ra = -1.0, +1.0
    else:
        outcome = "ongoing"
        rs, ra = 0.0, 0.0

    # Guard against empty row (no valid moves). Treat as mover loss.
    if len(cells) == 0 and outcome == "ongoing":
        # This should rarely happen. Let mover lose just like sum > 20.
        if mover == SHRINKER:
            outcome = "amplifier_wins"
            rs, ra = -1.0, +1.0
        else:
            outcome = "shrinker_wins"
            rs, ra = +1.0, -1.0

    return {"cells": cells, "move_num": move_num}, outcome, rs, ra


def render_state(state, outcome=None):
    cells  = state["cells"]
    player = current_player(state)
    name   = "Shrinker" if player == SHRINKER else "Amplifier"
    row    = "  ".join(f"[{c:>2}]" for c in cells)
    s      = f"  Move {state['move_num']:>2} | {row} | sum={sum(cells)} | Turn: {name}"
    if outcome and outcome != "ongoing":
        winner = "Shrinker" if "shrinker" in outcome else "Amplifier"
        s += f"\n  RESULT: {outcome} — {winner} wins!"
    return s


# ════════════════════════════════════════════════════════════════
#  Q-LEARNING AGENT (unchanged)
# ════════════════════════════════════════════════════════════════

class QAgent:
    def __init__(self, role, alpha=0.2, gamma=0.92,
                 eps=1.0, eps_min=0.05, eps_decay=0.9997):
        assert role in (SHRINKER, AMPLIFIER)
        self.role      = role
        self.alpha     = alpha
        self.gamma     = gamma
        self.eps       = eps
        self.eps_min   = eps_min
        self.eps_decay = eps_decay
        self.Q         = defaultdict(lambda: defaultdict(float))
        self.history   = {"episode": [], "win_rate": [], "epsilon": [], "avg_moves": []}

    def _key(self, state):
        return (tuple(state["cells"]), state["move_num"])

    def _flat(self, action):
        idx, op = action
        return idx * 2 + (0 if op == "amplify" else 1)

    def best_action(self, state):
        acts = valid_actions(state)
        return max(acts, key=lambda a: self.Q[self._key(state)][self._flat(a)])

    def act(self, state, explore=True):
        if explore and random.random() < self.eps:
            return random.choice(valid_actions(state))
        return self.best_action(state)

    def update(self, state, action, reward, next_state, done):
        key  = self._key(state)
        flat = self._flat(action)
        if done:
            target = reward
        else:
            acts   = valid_actions(next_state)
            future = max(self.Q[self._key(next_state)][self._flat(a)] for a in acts)
            target = reward + self.gamma * future
        self.Q[key][flat] += self.alpha * (target - self.Q[key][flat])

    def decay(self):
        self.eps = max(self.eps_min, self.eps * self.eps_decay)

    def save(self, path):
        data = {
            "role":      self.role,
            "alpha":     self.alpha,
            "gamma":     self.gamma,
            "eps":       self.eps,
            "eps_min":   self.eps_min,
            "eps_decay": self.eps_decay,
            "Q":         {str(k): dict(v) for k, v in self.Q.items()},
            "history":   self.history,
        }
        with open(path, "w") as f:
            json.dump(data, f)
        rname = "Shrinker" if self.role == SHRINKER else "Amplifier"
        print(f"  Saved {rname} -> {path}  ({len(self.Q)} states)")

    @classmethod
    def load(cls, path):
        with open(path) as f:
            data = json.load(f)
        agent = cls(
            role=data["role"], alpha=data["alpha"], gamma=data["gamma"],
            eps=data["eps"], eps_min=data["eps_min"], eps_decay=data["eps_decay"],
        )
        for k_str, v in data["Q"].items():
            k = eval(k_str)
            agent.Q[k] = defaultdict(float, {int(a): q for a, q in v.items()})
        agent.history = data["history"]
        rname = "Shrinker" if agent.role == SHRINKER else "Amplifier"
        print(f"  Loaded {rname} <- {path}  ({len(agent.Q)} states)")
        return agent


# ════════════════════════════════════════════════════════════════
#  BASELINE AGENTS (unchanged – they still play heuristically)
# ════════════════════════════════════════════════════════════════

class RandomAgent:
    def __init__(self, role): self.role = role
    def act(self, state, explore=True):
        return random.choice(valid_actions(state))


class HeuristicShrinker:
    """1-step greedy shrinker — mimics LLM reasoning from text."""
    role = SHRINKER
    def act(self, state, explore=True):
        best_a, best_s = None, -999
        cells = state["cells"]
        for a in valid_actions(state):
            ns, outcome, _, _ = transition(deepcopy(state), a)
            if "shrinker_wins" in outcome:    score = 1000
            elif "amplifier_wins" in outcome: score = -500
            else:
                score = (len(cells) - len(ns["cells"])) * 10 - sum(ns["cells"]) * 0.1
            if score > best_s:
                best_s, best_a = score, a
        return best_a


class HeuristicAmplifier:
    """1-step greedy amplifier — mimics LLM reasoning from text."""
    role = AMPLIFIER
    def act(self, state, explore=True):
        best_a, best_s = None, -999
        for a in valid_actions(state):
            ns, outcome, _, _ = transition(deepcopy(state), a)
            if "amplifier_wins" in outcome:  score = 1000
            elif "shrinker_wins" in outcome: score = -500
            else:
                score = sum(ns["cells"]) * 0.5 + len(ns["cells"]) * 2
            if score > best_s:
                best_s, best_a = score, a
        return best_a


# ════════════════════════════════════════════════════════════════
#  EPISODE RUNNER (unchanged, uses the fixed transition rewards)
# ════════════════════════════════════════════════════════════════

def run_episode(shrinker_agent, amplifier_agent,
                train_shrinker=False, train_amplifier=False):
    state   = initial_state()
    s_traj  = []
    a_traj  = []
    outcome = "timeout"

    for _ in range(60):
        player = current_player(state)
        agent  = shrinker_agent if player == SHRINKER else amplifier_agent
        action = agent.act(state, explore=(train_shrinker or train_amplifier))

        if player == SHRINKER:
            s_traj.append((deepcopy(state), action))
        else:
            a_traj.append((deepcopy(state), action))

        next_state, outcome, rs, ra = transition(deepcopy(state), action)
        state = next_state
        if outcome != "ongoing":
            break

    s_won = "shrinker"  in outcome
    a_won = "amplifier" in outcome

    if train_shrinker and isinstance(shrinker_agent, QAgent):
        final_r = +1.0 if s_won else -1.0
        for i, (s, a) in enumerate(s_traj):
            is_last = (i == len(s_traj) - 1)
            ns = s_traj[i+1][0] if not is_last else state
            shrinker_agent.update(s, a, final_r if is_last else 0.0, ns, is_last)

    if train_amplifier and isinstance(amplifier_agent, QAgent):
        final_r = +1.0 if a_won else -1.0
        for i, (s, a) in enumerate(a_traj):
            is_last = (i == len(a_traj) - 1)
            ns = a_traj[i+1][0] if not is_last else state
            amplifier_agent.update(s, a, final_r if is_last else 0.0, ns, is_last)

    return outcome, state["move_num"]


# ════════════════════════════════════════════════════════════════
#  EVALUATION (unchanged)
# ════════════════════════════════════════════════════════════════

def evaluate(shrinker_agent, amplifier_agent, role, n=500):
    wins = 0; total_moves = 0
    for _ in range(n):
        outcome, moves = run_episode(shrinker_agent, amplifier_agent)
        total_moves += moves
        if role == SHRINKER and "shrinker" in outcome:
            wins += 1
        elif role == AMPLIFIER and "amplifier" in outcome:
            wins += 1
    return wins / n, total_moves / n


# ════════════════════════════════════════════════════════════════
#  TRAINING (unchanged)
# ════════════════════════════════════════════════════════════════

def train(
    n_episodes     = 30_000,
    eval_interval  = 1_000,
    eval_n         = 500,
    shrinker_path  = "/mnt/user-data/outputs/flux_shrinker.json",
    amplifier_path = "/mnt/user-data/outputs/flux_amplifier.json",
    plot_path      = "/mnt/user-data/outputs/flux_training.png",
):
    print("=" * 64)
    print("  FLUX — Dual Agent RL Training")
    print("=" * 64)
    print(f"  Episodes : {n_episodes:,}  |  eval every {eval_interval}")
    print()

    shrinker  = QAgent(role=SHRINKER)
    amplifier = QAgent(role=AMPLIFIER)
    rand_s    = RandomAgent(SHRINKER)
    rand_a    = RandomAgent(AMPLIFIER)

    metrics = {
        "ep": [], "s_win": [], "a_win": [],
        "cross_s": [], "eps_s": [], "eps_a": [], "avg_moves": [],
    }

    # Random baseline (one-time)
    s_base, _ = evaluate(rand_s, rand_a, SHRINKER, n=2000)
    a_base, _ = evaluate(rand_s, rand_a, AMPLIFIER, n=2000)
    print(f"  Random baselines — Shrinker: {s_base:.1%}  Amplifier: {a_base:.1%}\n")

    t0 = time.time()

    for ep in range(1, n_episodes + 1):
        # Three training modes, cycling
        mode = ep % 3
        if mode == 0:
            run_episode(shrinker, rand_a,    train_shrinker=True)
        elif mode == 1:
            run_episode(rand_s,   amplifier, train_amplifier=True)
        else:
            run_episode(shrinker, amplifier,
                        train_shrinker=True, train_amplifier=True)

        shrinker.decay()
        amplifier.decay()

        if ep % eval_interval == 0:
            s_wr, s_mv = evaluate(shrinker, rand_a,    SHRINKER,  eval_n)
            a_wr, a_mv = evaluate(rand_s,   amplifier, AMPLIFIER, eval_n)
            cr_s, _    = evaluate(shrinker, amplifier, SHRINKER,  eval_n)

            metrics["ep"].append(ep)
            metrics["s_win"].append(s_wr)
            metrics["a_win"].append(a_wr)
            metrics["cross_s"].append(cr_s)
            metrics["eps_s"].append(shrinker.eps)
            metrics["eps_a"].append(amplifier.eps)
            metrics["avg_moves"].append((s_mv + a_mv) / 2)

            for h, v in [("episode", ep), ("win_rate", s_wr),
                         ("epsilon", shrinker.eps), ("avg_moves", s_mv)]:
                shrinker.history[h].append(v)
            for h, v in [("episode", ep), ("win_rate", a_wr),
                         ("epsilon", amplifier.eps), ("avg_moves", a_mv)]:
                amplifier.history[h].append(v)

            print(
                f"  ep {ep:>6,} | eps S={shrinker.eps:.3f} A={amplifier.eps:.3f} | "
                f"S_win={s_wr:.1%} A_win={a_wr:.1%} cross_S={cr_s:.1%} | "
                f"states S={len(shrinker.Q)} A={len(amplifier.Q)} | "
                f"{time.time()-t0:.0f}s"
            )

    print(f"\n  Done in {time.time()-t0:.1f}s")
    shrinker.save(shrinker_path)
    amplifier.save(amplifier_path)
    _plot(metrics, s_base, a_base, plot_path)
    return shrinker, amplifier


# ════════════════════════════════════════════════════════════════
#  PLOT (unchanged)
# ════════════════════════════════════════════════════════════════

def _plot(metrics, s_base, a_base, path):
    eps = metrics["ep"]
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    fig.suptitle("FLUX — RL Training Metrics", fontsize=13, fontweight="bold")

    ax = axes[0, 0]
    ax.plot(eps, metrics["s_win"],  color="#185FA5", lw=2, label="Shrinker vs random")
    ax.plot(eps, metrics["a_win"],  color="#993556", lw=2, label="Amplifier vs random")
    ax.plot(eps, metrics["cross_s"], color="#7F77DD", lw=1.5, ls="--", label="Shrinker in cross-play")
    ax.axhline(s_base, color="#185FA5", lw=0.8, ls=":", alpha=0.5, label=f"S baseline {s_base:.0%}")
    ax.axhline(a_base, color="#993556", lw=0.8, ls=":", alpha=0.5, label=f"A baseline {a_base:.0%}")
    ax.set_title("Win rates"); ax.set_xlabel("Episode"); ax.set_ylabel("Win rate")
    ax.set_ylim(0, 1.05); ax.legend(fontsize=8); ax.grid(alpha=0.25)

    ax = axes[0, 1]
    ax.plot(eps, metrics["eps_s"], color="#185FA5", lw=2, label="Shrinker ε")
    ax.plot(eps, metrics["eps_a"], color="#993556", lw=2, label="Amplifier ε")
    ax.set_title("Exploration (ε)"); ax.set_xlabel("Episode"); ax.set_ylabel("Epsilon")
    ax.set_ylim(0, 1.05); ax.legend(); ax.grid(alpha=0.25)

    ax = axes[1, 0]
    ax.plot(eps, metrics["avg_moves"], color="#0F6E56", lw=2)
    ax.axhline(MAX_MOVES, color="#aaa", lw=1, ls="--", label="Tiebreak at 15")
    ax.set_title("Avg game length"); ax.set_xlabel("Episode"); ax.set_ylabel("Moves")
    ax.legend(); ax.grid(alpha=0.25)

    ax = axes[1, 1]
    s_gap = [w - s_base for w in metrics["s_win"]]
    a_gap = [w - a_base for w in metrics["a_win"]]
    ax.plot(eps, s_gap, color="#185FA5", lw=2, label="Shrinker gain")
    ax.plot(eps, a_gap, color="#993556", lw=2, label="Amplifier gain")
    ax.axhline(0, color="#aaa", lw=1, ls="--", label="Random baseline")
    ax.set_title("Improvement over random baseline")
    ax.set_xlabel("Episode"); ax.set_ylabel("Win rate gain")
    ax.legend(); ax.grid(alpha=0.25)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:+.0%}"))

    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Plot saved -> {path}")


# ════════════════════════════════════════════════════════════════
#  FULL REPORT (unchanged)
# ════════════════════════════════════════════════════════════════

def full_report(shrinker, amplifier, n=1000):
    rand_s  = RandomAgent(SHRINKER)
    rand_a  = RandomAgent(AMPLIFIER)
    heur_s  = HeuristicShrinker()
    heur_a  = HeuristicAmplifier()

    print("\n" + "=" * 64)
    print(f"  FULL EVALUATION REPORT  ({n:,} games each)")
    print("=" * 64)

    rows = []
    configs = [
        ("Random  Shrinker vs Random  Amplifier", rand_s,     rand_a,     SHRINKER),
        ("Heurist Shrinker vs Random  Amplifier", heur_s,     rand_a,     SHRINKER),
        ("RL      Shrinker vs Random  Amplifier", shrinker,   rand_a,     SHRINKER),
        ("RL      Shrinker vs Heurist Amplifier", shrinker,   heur_a,     SHRINKER),
        ("Random  Shrinker vs Random  Amplifier", rand_s,     rand_a,     AMPLIFIER),
        ("Random  Shrinker vs Heurist Amplifier", rand_s,     heur_a,     AMPLIFIER),
        ("Random  Shrinker vs RL      Amplifier", rand_s,     amplifier,  AMPLIFIER),
        ("Heurist Shrinker vs RL      Amplifier", heur_s,     amplifier,  AMPLIFIER),
        ("RL      Shrinker vs RL      Amplifier", shrinker,   amplifier,  SHRINKER),
    ]

    print(f"\n  {'Matchup':<45} {'Role':<10} {'Win%':>6}  {'Moves':>6}")
    print("  " + "-" * 70)
    for label, sa, aa, role in configs:
        wr, mv = evaluate(sa, aa, role, n)
        rname  = "Shrinker" if role == SHRINKER else "Amplifier"
        marker = " <-- cross-play" if "RL" in label and "RL" in label[label.index("vs"):] else ""
        print(f"  {label:<45} {rname:<10} {wr:>6.1%}  {mv:>6.1f}{marker}")


    print()
    """
    print("  LLM-style vs World-Model Agent summary:")
    s_heur_wr, _ = evaluate(heur_s,   rand_a,    SHRINKER,  n)
    s_rl_wr,   _ = evaluate(shrinker, heur_a,    SHRINKER,  n)
    a_heur_wr, _ = evaluate(rand_s,   heur_a,    AMPLIFIER, n)
    a_rl_wr,   _ = evaluate(heur_s,   amplifier, AMPLIFIER, n)
    print(f"  Shrinker  | LLM-heuristic: {s_heur_wr:.1%}  RL agent: {s_rl_wr:.1%}  "
          f"gain: {(s_rl_wr-s_heur_wr)*100:+.1f}pp")
    print(f"  Amplifier | LLM-heuristic: {a_heur_wr:.1%}  RL agent: {a_rl_wr:.1%}  "
          f"gain: {(a_rl_wr-a_heur_wr)*100:+.1f}pp")
    print()
    print("  The gain above is the empirical signature of:")
    print("  latent state modeling (S) > observation prediction (X)")
    print("=" * 64)
    """

# ════════════════════════════════════════════════════════════════
#  HUMAN PLAY (unchanged)
# ════════════════════════════════════════════════════════════════

def play_human(shrinker_agent, amplifier_agent):
    print("\n" + "=" * 64)
    print("  FLUX — Human vs Trained Agent")
    print("=" * 64)
    print("\n  Row starts: [2, 1, 3, 1, 2]")
    print("  Shrinker wins: row -> 1 cell")
    print(f"  Amplifier wins: sum > {SUM_CAP}")
    print(f"  Tiebreak at move {MAX_MOVES}: <{TIEBREAK_CELLS} cells -> Shrinker wins")

    ri = input("\n  Play as Shrinker [0] or Amplifier [1]? ").strip()
    human_role = int(ri) if ri in ("0","1") else 0
    rname = "Shrinker" if human_role == SHRINKER else "Amplifier"
    print(f"\n  You are the {rname}.\n")

    state   = initial_state()
    outcome = "ongoing"

    while outcome == "ongoing":
        player = current_player(state)
        print(render_state(state))

        if player == human_role:
            acts = valid_actions(state)
            for i, (ci, op) in enumerate(acts):
                print(f"  [{i:>2}] {op:8} cell {ci} (={state['cells'][ci]})")
            while True:
                try:
                    ch = int(input("  Choose: "))
                    if 0 <= ch < len(acts):
                        action = acts[ch]; break
                except (ValueError, EOFError):
                    pass
                print("  Invalid.")
        else:
            agent  = shrinker_agent if player == SHRINKER else amplifier_agent
            action = agent.act(state, explore=False)
            ai_name = "Shrinker-AI" if player == SHRINKER else "Amplifier-AI"
            idx, op = action
            print(f"  {ai_name}: {op} cell {idx} (={state['cells'][idx]})")

        state, outcome, _, _ = transition(deepcopy(state), action)
        print()

    print(render_state(state, outcome))
    winner = "Shrinker" if "shrinker" in outcome else "Amplifier"
    print("\n  You win!" if winner == rname else "\n  Agent wins.")


# ════════════════════════════════════════════════════════════════
#  LLM INTERFACE (unchanged)
# ════════════════════════════════════════════════════════════════

def state_to_prompt(state):
    """P(x|s): project latent state to natural language for LLM."""
    cells  = state["cells"]
    player = current_player(state)
    role   = "Shrinker" if player == SHRINKER else "Amplifier"
    goal   = ("reduce the row to 1 cell"
              if player == SHRINKER else f"push sum above {SUM_CAP}")
    return (
        f"FLUX — move {state['move_num']+1}.\n"
        f"Row: {cells}. Sum: {sum(cells)}. Moves left: {MAX_MOVES - state['move_num']}.\n"
        f"You are the {role}. Goal: {goal}.\n"
        f"Choose: INDEX (0-based) + OPERATION (amplify or drain). "
        f"Reply exactly as: INDEX OPERATION"
    )


def parse_llm_move(text, n_cells):
    parts = text.strip().lower().split()
    try:
        idx = int(parts[0])
        op  = parts[1] if len(parts) > 1 else ""
        if op in ("amplify", "drain") and 0 <= idx < n_cells:
            return (idx, op)
    except (IndexError, ValueError):
        pass
    return None


def llm_sim(shrinker, amplifier, n=500):
    """Show sample prompts + run simulation comparing RL vs heuristic."""
    print("\n" + "=" * 64)
    print("  LLM Interface Demo + Comparison")
    print("=" * 64)

    # Sample prompt
    state = initial_state()
    print("\n  Sample prompt (LLM receives this as text):")
    print("  " + "─" * 50)
    for line in state_to_prompt(state).split("\n"):
        print(f"    {line}")
    print("  " + "─" * 50)

    # Simulate
    heur_s = HeuristicShrinker()
    heur_a = HeuristicAmplifier()
    s_rl_wins, s_llm_wins = 0, 0
    for _ in range(n):
        o, _ = run_episode(shrinker, heur_a)
        if "shrinker" in o: s_rl_wins += 1
        else: s_llm_wins += 1

    a_rl_wins, a_llm_wins = 0, 0
    for _ in range(n):
        o, _ = run_episode(heur_s, amplifier)
        if "amplifier" in o: a_rl_wins += 1
        else: a_llm_wins += 1

    print(f"\n  RL Shrinker vs LLM-style Amplifier ({n} games):")
    print(f"    RL  wins: {s_rl_wins/n:.1%}   LLM wins: {s_llm_wins/n:.1%}")
    print(f"\n  LLM-style Shrinker vs RL Amplifier ({n} games):")
    print(f"    RL  wins: {a_rl_wins/n:.1%}   LLM wins: {a_llm_wins/n:.1%}")
    print()
    print("  Gap = empirical cost of operating in X vs S")
    print("=" * 64)


# ════════════════════════════════════════════════════════════════
#  MAIN
# ════════════════════════════════════════════════════════════════

SP = "flux_shrinker.json"
AP = "flux_amplifier.json"
PP = "flux_training.png"


def main():
    print("\n  FLUX — World Model + RL")
    print("  1) Train   2) Load + Report   3) Load + Play   4) Load + LLM demo")
    mode = input("  [1/2/3/4] default=1: ").strip() or "1"

    if mode == "1":
        s, a = train(n_episodes=30_000, eval_interval=1_000, eval_n=500,
                     shrinker_path=SP, amplifier_path=AP, plot_path=PP)
        full_report(s, a)
        llm_sim(s, a)
    elif mode in ("2","3","4"):
        s = QAgent.load(SP)
        a = QAgent.load(AP)
        if mode == "2": full_report(s, a)
        elif mode == "3": play_human(s, a)
        elif mode == "4": llm_sim(s, a)


if __name__ == "__main__":
    main()


  FLUX — World Model + RL
  1) Train   2) Load + Report   3) Load + Play   4) Load + LLM demo
  [1/2/3/4] default=1: 2
  Loaded Shrinker <- flux_shrinker.json  (3051 states)
  Loaded Amplifier <- flux_amplifier.json  (2718 states)

  FULL EVALUATION REPORT  (1,000 games each)

  Matchup                                       Role         Win%   Moves
  ----------------------------------------------------------------------
  Random  Shrinker vs Random  Amplifier         Shrinker    43.3%    12.3
  Heurist Shrinker vs Random  Amplifier         Shrinker    77.6%    10.2
  RL      Shrinker vs Random  Amplifier         Shrinker    89.5%    11.1
  RL      Shrinker vs Heurist Amplifier         Shrinker     0.0%    13.0
  Random  Shrinker vs Random  Amplifier         Amplifier   57.4%    12.0
  Random  Shrinker vs Heurist Amplifier         Amplifier   99.5%     8.8
  Random  Shrinker vs RL      Amplifier         Amplifier   98.8%    11.6 <-- cross-play
  Heurist Shrinker vs RL      Amplifier  